# Living-park cross-sectional analysis

## Compute CSV files for area, volume and thickness measurements

In [41]:
import glob
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

import joblib
import numpy as np
from freesurfer.aparcstats2table import main as aparcstats2table
from freesurfer.asegstats2table import main as asegstats2table

root_dir = os.path.dirname(os.getcwd())  # Use current working directory as root_dir
display("Running in root dir: ", os.path.basename(root_dir))

'Running in root dir: '

'notebooks'

### Build TSV tables from raw data

In [39]:
dataset = pd.read_json("../json/patno_ids_with_more_than_25_successful_recon_all.json")

In [ ]:
measurements = ['volume', 'thickness', 'area']
input_dir = os.path.realpath(os.path.join(root_dir, "../vip_outputs"))
output_dir = 'tables_QCed'
os.makedirs(output_dir, exist_ok=True)

rsv = {f"rep{rep}": dataset[dataset["repetition"] == rep]["subject_visit"].tolist() for rep in dataset["repetition"].unique()}

def compute_aseg_volume(subjects_dir, repetition, subjects, output_dir):
    subjects_dir = os.path.join(subjects_dir, repetition)
    tablefile = os.path.join(output_dir, f"{repetition}.aseg.volume.tsv")
    args = [f"--sd={subjects_dir}",
            "--skip",
            "--subjects",
            *subjects,
            "--meas=volume",
            f"--tablefile={tablefile}"
    ]
    asegstats2table(args)

def compute_volume(subjects_dir, repetition, subjects, output_dir):
    subjects_dir = os.path.join(subjects_dir, repetition)
    for hemi in ["lh", "rh"]:
        tablefile = os.path.join(output_dir, f"{repetition}.{hemi}.aparc.volume.tsv")
        args = [
            f"--sd={subjects_dir}",
            "--skip",
            "--parc=aparc.a2009s",
            f"--hemi={hemi}",
            "--subjects",
            *subjects,
            "--meas=volume",
            f"--tablefile={tablefile}",
        ]
        aparcstats2table(args)

def compute_thickness(subjects_dir, repetition, subjects, output_dir):
    subjects_dir = os.path.join(subjects_dir, repetition)
    for hemi in ["lh", "rh"]:
        tablefile = os.path.join(output_dir, f"{repetition}.{hemi}.aparc.thickness.tsv")
        args = [
            f"--sd={subjects_dir}",
            "--skip",
            "--parc=aparc.a2009s",
            f"--hemi={hemi}",
            "--subjects",
            *subjects,
            "--meas=thickness",
            f"--tablefile={tablefile}"
        ]
        aparcstats2table(args)

def compute_area(subjects_dir, repetition, subjects, output_dir):
    subjects_dir = os.path.join(subjects_dir, repetition)
    for hemi in ["lh", "rh"]:
        tablefile = os.path.join(output_dir, f"{repetition}.{hemi}.aparc.area.tsv")
        args = [
            f"--sd={subjects_dir}",
            "--skip",
            "--parc=aparc.a2009s",
            f"--hemi={hemi}",
            "--subjects",
            *subjects,
            "--meas=area",
            f"--tablefile={tablefile}"
        ]
        aparcstats2table(args)


joblib.Parallel(n_jobs=-1, verbose=10)(joblib.delayed(compute_aseg_volume)(input_dir, repetition, subjects, output_dir) for repetition, subjects in rsv.items())
joblib.Parallel(n_jobs=-1, verbose=10)(
    joblib.delayed(compute_volume)(input_dir, repetition, subjects, output_dir)
    for repetition, subjects in rsv.items()
)
joblib.Parallel(n_jobs=-1, verbose=10)(
    joblib.delayed(compute_thickness)(input_dir, repetition, subjects, output_dir)
    for repetition, subjects in rsv.items()
)
joblib.Parallel(n_jobs=-1, verbose=10)(
    joblib.delayed(compute_area)(input_dir, repetition, subjects, output_dir)
    for repetition, subjects in rsv.items()
);

### Read TSV tables

In [70]:
tsv_tables = glob.glob('../tables_QCed/*.tsv')
aseg_tables_group = {}
aparc_tables_group = {}
for tsv_table in tsv_tables:
    fields = os.path.basename(tsv_table).split('.')
    if fields[1] == "aseg":
        aseg_tables_group["volume"] = aseg_tables_group.get("volume", []) + [tsv_table]
    elif fields[1] in ["lh", "rh"]:
        hemi = fields[1]
        measure = fields[3]
        aparc_tables_group[measure] = aparc_tables_group.get(measure, {})
        aparc_tables_group[measure][hemi] = aparc_tables_group[measure].get(hemi, []) + [tsv_table]

In [85]:
print("Parcellation tables found:")
for measure, tables in aparc_tables_group.items():
    for hemi, tsv_tables in tables.items():
        print(f"\tFound {len(tsv_tables)} for {measure} {hemi} tables")
print("Segmentation tables found:")
for measure, tsv_tables in aseg_tables_group.items():
    print(f"\tFound {len(tsv_tables)} for {measure} tables")        

Parcellation tables found:
	Found 34 for area lh tables
	Found 34 for area rh tables
	Found 34 for volume lh tables
	Found 34 for volume rh tables
	Found 34 for thickness rh tables
	Found 34 for thickness lh tables
Segmentation tables found:
	Found 34 for volume tables


### Merge tables 

In [78]:
stats_dir = Path(root_dir) / "stats_QCed"
raw_stats_dir = stats_dir / "raw"
os.makedirs(stats_dir, exist_ok=True)
os.makedirs(raw_stats_dir, exist_ok=True)

In [96]:
import pandas as pd

def read_tsv(tsv):
    df = pd.read_csv(tsv, sep="\t")
    df.rename(lambda column: column.replace("lh_",""), inplace=True, axis=1)
    df.rename(lambda column: column.replace("rh_",""), inplace=True, axis=1)
    df.rename(columns={df.columns[0]: "subjects"}, inplace=True)
    if '.lh.' in tsv:
        df["hemi"] = "lh"
    elif '.rh.' in tsv:
        df["hemi"] = "rh"
    return df

def concat_tables(tables, hemi=True):
    if hemi:
        lh = pd.concat([read_tsv(table) for table in tables["lh"]])
        rh = pd.concat([read_tsv(table) for table in tables["rh"]])        
        concat = pd.concat((lh, rh))
        return concat
    else:
        return pd.concat([read_tsv(table) for table in tables])

thickness_df = concat_tables(aparc_tables_group["thickness"])
thickness_df.to_parquet(raw_stats_dir / "thickness.parquet")

area_df = concat_tables(aparc_tables_group["area"])
area_df.to_parquet(raw_stats_dir  / "area.parquet")

volume_df = concat_tables(aparc_tables_group["volume"])
volume_df.to_parquet(raw_stats_dir / "volume.parquet")

subcortical_volume = concat_tables(aseg_tables_group["volume"], hemi=False)
subcortical_volume.to_parquet(raw_stats_dir  / "subcortical_volume.parquet")

### Load pre-computed tables

In [129]:
thickness_df = pd.read_parquet(raw_stats_dir / "thickness.parquet")
area_df = pd.read_parquet(raw_stats_dir / "area.parquet")
volume_df = pd.read_parquet(raw_stats_dir / "volume.parquet")
subcortical_volume_df = pd.read_parquet(raw_stats_dir / "subcortical_volume.parquet")

In [136]:
def keep_first_rows(df, n=25, hemi=None):
    """
    Keep the first n rows for each subject
    """
    print(f"Keeping {n} rows for each subject hemi: {hemi}")
    sort_keys = ["subjects"] + (["hemi"] if hemi else [])
    # Keep n-th first rows for each subjects
    if hemi:
        lh = df[df["hemi"] == "lh"].sort_values(by=sort_keys).groupby("subjects").head(n)
        rh = df[df["hemi"] == "rh"].sort_values(by=sort_keys).groupby("subjects").head(n)
        df_sampled = pd.concat((lh, rh))
    else:
        df_sampled = df.sort_values(by=sort_keys).groupby("subjects").head(n)

    # Assert that the number of rows is 25*number of subjects
    if hemi:
        count_sample = (df_sampled.groupby(["subjects","hemi"]).count() == n).reset_index()
        less_than_n  = count_sample[count_sample["eTIV"] == False]
        # remove subjects with less than n rows
        if len(less_than_n) > 0:
            not_n_subjects = less_than_n["subjects"].unique()
            df_sampled = df_sampled[~df_sampled["subjects"].isin(not_n_subjects)]

            # Encoded name to avoid printing the full name
            encoded_subjects = [hash(subject) for subject in not_n_subjects]
            print(f"Subjects with less than {n} rows: {encoded_subjects}")

        nb_rows_lh = len(df_sampled[df_sampled["hemi"] == "lh"])
        nb_rows_rh = len(df_sampled[df_sampled["hemi"] == "rh"])

        nb_subjects_lh = len(df_sampled[df_sampled["hemi"] == "lh"]["subjects"].unique())
        nb_subjects_rh = len(df_sampled[df_sampled["hemi"] == "rh"]["subjects"].unique())

        assert nb_rows_lh == n * nb_subjects_lh
        assert nb_rows_rh == n * nb_subjects_rh

        print(f"Number of subjects: {len(df_sampled['subjects'].unique())}")
    else:
        count_sample = (df_sampled.groupby(["subjects"]).count() == n).reset_index()
        less_than_n = count_sample[
            count_sample["EstimatedTotalIntraCranialVol"] == False
        ]
        # remove subjects with less than n rows
        if len(less_than_n) > 0:
            not_n_subjects = less_than_n["subjects"].unique()
            df_sampled = df_sampled[~df_sampled["subjects"].isin(not_n_subjects)]

            # Encoded name to avoid printing the full name
            encoded_subjects = [hash(subject) for subject in not_n_subjects]
            print(f"Subjects with less than {n} rows: {encoded_subjects}")

        assert len(df_sampled) == n * len(df_sampled["subjects"].unique())    
        print(f"Number of subjects: {len(df_sampled['subjects'].unique())}")

    return df_sampled

print("Keeping 25 repetitions for each subject")
print("Cortical thickness")
thickness_sampled_df = keep_first_rows(thickness_df, n=25, hemi=True)
print("Cortical area")
area_sampled_df = keep_first_rows(area_df, n=25, hemi=True)
print("Cortical volume")
volume_sampled_df = keep_first_rows(volume_df, n=25, hemi=True)
print("Subcortical volume")
subcortical_volume_sampled_df = keep_first_rows(subcortical_volume_df, n=25, hemi=False)

Keeping 25 repetitions for each subject
Cortical thickness
Keeping 25 rows for each subject hemi: True
Subjects with less than 25 rows: [8954594018897395270, -4726226401682941732, 452083929236321718, -6070298443988882697, 4695266586701530376, -4373669357560828164, -5591163590156027987, 4057970555135467265, 5105249996339310521, -7407800758811024876]
Number of subjects: 600
Cortical area
Keeping 25 rows for each subject hemi: True
Subjects with less than 25 rows: [8954594018897395270, -4726226401682941732, 452083929236321718, -6070298443988882697, 4695266586701530376, -4373669357560828164, -5591163590156027987, 4057970555135467265, 5105249996339310521, -7407800758811024876]
Number of subjects: 600
Cortical volume
Keeping 25 rows for each subject hemi: True
Subjects with less than 25 rows: [8954594018897395270, -4726226401682941732, 452083929236321718, -6070298443988882697, 4695266586701530376, -4373669357560828164, -5591163590156027987, 4057970555135467265, 5105249996339310521, -74078007

### Save 

In [137]:
sampled_stats_dir = stats_dir / "sampled"
os.makedirs(sampled_stats_dir, exist_ok=True)

description = {
    "sample_size": 25,
    "thickness": "Cortical thickness",
    "area": "Cortical area",
    "volume": "Cortical volume",
    "subcortical_volume": "Subcortical volume"
}
with open(sampled_stats_dir / "description.json", "w") as f:
    json.dump(description, f, indent=4)

thickness_sampled_df.to_parquet(sampled_stats_dir / "thickness.parquet")
area_sampled_df.to_parquet(sampled_stats_dir / "area.parquet")
volume_sampled_df.to_parquet(sampled_stats_dir / "volume.parquet")
subcortical_volume_sampled_df.to_parquet(sampled_stats_dir / "subcortical_volume.parquet")